# XM655 self interference cancellation without a channel estimate

Two stage BSIC, both sides full duplex, designed on **nothing but received
blocks** - no stored channels, no `channel_estimation.ipynb` before it. The
board is trained the way a real link would be, in solo slots, and then tested with
both sides on air at once.

```
tones --> first training --> tx weights --> second training --> rx weights
                                                                      |
                             measurements <-- FFT <-- test capture <--+
```

- **Stage 1** - each side probes its own self interference channel one DAC at a
  time, builds the gram matrix `H^H H` out of what came back, and transmits down
  its **quietest** eigenvector. Least power leaks onto its own receivers, so its
  amplifier has the least to survive.
- **Stage 2** - each side transmits alone with that weight on air. Its own ADCs
  hold the leakage that is left, the other side's ADCs hold the link. The receive
  weight is the strongest generalized eigenvector of the two covariances - the
  most wanted power per unit of what still leaks.

Same algorithm as `no_channel_algo_testing.py` in the simulation, with the
`sim.get_channel()` calls replaced by captures. No digital canceller here.

Each side transmits its own tone, so one FFT on one receiver reads the leakage
and the link on two different bins.

## 1. Parameters

Everything that gets tuned. Nothing outside this cell should need editing.

Side 0 is the simulation's base station, side 1 its user. Both are full duplex
here, so the two are symmetric and the roles are only names.

In [ ]:
from lib.config_parser import load_config

CFG = load_config()

# --- side 0 ---
DACS_0 = CFG["systems"]["dacs_0"]    # overlay.dac[] indices, tile 2
ADCS_0 = CFG["systems"]["adcs_0"]    # its own receivers, tile 1

# --- side 1 ---
DACS_1 = CFG["systems"]["dacs_1"]    # overlay.dac[] indices, tile 3
ADCS_1 = CFG["systems"]["adcs_1"]    # its own receivers, tile 0

# --- rf ---
DAC_NCO  = CFG["rf"]["dac_nco"]      # MHz, per tile -> TX lands at 4900
DAC_ZONE = CFG["rf"]["dac_zone"]     # Nyquist zone, per tile
ADC_NCO  = CFG["rf"]["adc_nco"]      # MHz, per tile = 5000 - DAC_NCO
ADC_ZONE = CFG["rf"]["adc_zone"]     # the fold is in an even zone

# --- rates: fixed by the bitstream, do not change ---
DAC_SR = CFG["board"]["dac_sr"]      # DAC baseband rate = 10 GSPS / 10 (C2R eats one x2)
ADC_SR = CFG["board"]["adc_sr"]      # ADC baseband rate = 2.5 GSPS / 10 decimation
N_CH   = CFG["board"]["n_ch"]        # RF channels on the XM655
N_TILE = CFG["board"]["n_tile"]      # ADC and DAC tiles

PATH_PER_TILE = N_CH // N_TILE       # converters per tile - one player memory each

# --- one tone per side, so the FFT can tell them apart ---
CW_TONE_0_MHZ = CFG["signal"]["tone_0_mhz"]  # side 0 transmits this
CW_TONE_1_MHZ = CFG["signal"]["tone_1_mhz"]  # side 1 transmits this
CW_AMP        = CFG["signal"]["amp"]         # 14 bit DAC: +16383 / -16384

# --- bsic ---
REGULARIZATION = CFG["bsic"]["regularization"]  # stage 2 lambda, added to the si covariance, in ADC counts^2
BSIC_MAX_GAIN  = 1.0        # ceiling on any element gain, the register allows 2.0
MAX_PHASE_DEG  = 179.99     # the converter rejects exactly 180, so clamp to this

# --- one block of samples ---
N_CAP       = CFG["capture"]["n_cap"]        # samples per channel
TRIG_HOLD_S = CFG["capture"]["trig_hold_s"]  # trig_cap must stay high for a whole capture window

# --- where this run is written ---
SIC_DIR = "output/no_channel_sic"

## 2. Verify the parameters

Catch a bad settings cell before anything touches the board.

`snap_tone_to_fft_bin()` overrides a parameter instead of rejecting it: every
correlation and every power below is read against a tone that has to sit exactly
on a bin, and the two tones have to sit on bins of their own.

In [ ]:
import os

import numpy as np

from lib.fd.non_joint_sic_bsic import find_max_generalized_eigenvector
from lib.common_functions import capture_aligned
from lib.common_functions import clear_dir
from lib.common_functions import convert_raw_to_iq
from lib.common_functions import create_tone_samples
from lib.common_functions import find_tone_bin
from lib.common_functions import save_json_params
from lib.common_functions import snap_tone_to_fft_bin
from lib.common_functions import tune_adcs
from lib.common_functions import tune_dacs
from lib.common_functions import write_tone_to_tile_player

def validate_systems():
    """Both sides are complete and disjoint, and each transmits from one DAC tile.

    Stage 2 needs a receive null, so a side has to have at least two receivers.
    """
    for name, indices in (("DACS_0", DACS_0), ("ADCS_0", ADCS_0),
                          ("DACS_1", DACS_1), ("ADCS_1", ADCS_1)):
        if len(indices) == 0:
            raise ValueError("%s is empty - both sides transmit and receive here" % name)
        if not set(indices) <= set(range(N_CH)):
            raise ValueError("%s holds indices outside 0..%d" % (name, N_CH - 1))
        if len(set(indices)) != len(indices):
            raise ValueError("%s lists the same index twice" % name)
    for name, adcs in (("ADCS_0", ADCS_0), ("ADCS_1", ADCS_1)):
        if len(adcs) < 2:
            raise ValueError("%s has one receiver - nothing to steer a null with" % name)
    shared_dacs = set(DACS_0) & set(DACS_1)
    if shared_dacs:
        raise ValueError("DACs %s are listed under both sides" % sorted(shared_dacs))
    shared_adcs = set(ADCS_0) & set(ADCS_1)
    if shared_adcs:
        raise ValueError("ADCs %s are listed under both sides" % sorted(shared_adcs))
    tile_of = {}
    for name, dacs in (("DACS_0", DACS_0), ("DACS_1", DACS_1)):
        tiles = sorted(set(dac // PATH_PER_TILE for dac in dacs))
        if len(tiles) != 1:
            raise ValueError("%s spans DAC tiles %s - there is one player memory per "
                             "tile, so a side sending its own tone must sit inside one"
                             % (name, tiles))
        tile_of[name] = tiles[0]
    if tile_of["DACS_0"] == tile_of["DACS_1"]:
        raise ValueError("both sides transmit from DAC tile %d - they would share one "
                         "player memory and could not send different tones"
                         % tile_of["DACS_0"])

def validate_rf_settings():
    """Converter tables are per tile, the folds add up, and both tones fit the band."""
    for name, table in (("DAC_NCO", DAC_NCO), ("DAC_ZONE", DAC_ZONE),
                        ("ADC_NCO", ADC_NCO), ("ADC_ZONE", ADC_ZONE)):
        if len(table) != N_TILE:
            raise ValueError("%s needs one entry per tile (%d), got %d"
                             % (name, N_TILE, len(table)))
    fold_mhz = 2 * (10 * ADC_SR / 1e6)
    for tile, (dac_nco, adc_nco) in enumerate(zip(DAC_NCO, ADC_NCO)):
        if abs(dac_nco + adc_nco - fold_mhz) > 1e-6:
            raise ValueError("tile %d: ADC_NCO should be %g - DAC_NCO"
                             % (tile, fold_mhz))
    for name, tone_mhz in (("CW_TONE_0_MHZ", CW_TONE_0_MHZ),
                           ("CW_TONE_1_MHZ", CW_TONE_1_MHZ)):
        if not 0 < tone_mhz < ADC_SR / 2e6:
            raise ValueError("%s must be between 0 and %g, got %g"
                             % (name, ADC_SR / 2e6, tone_mhz))
    if not 0 < CW_AMP <= 16383:
        raise ValueError("CW_AMP must be between 1 and 16383")
    if N_CAP <= 0 or N_CAP & (N_CAP - 1):
        raise ValueError("N_CAP must be a positive power of two, got %d" % N_CAP)
    if TRIG_HOLD_S <= 0:
        raise ValueError("TRIG_HOLD_S must be positive")

def validate_tone_separation():
    """The two tones sit on bins of their own, or neither can be read alone."""
    bin_0 = find_tone_bin(CW_TONE_0_MHZ, N_CAP, ADC_SR)
    bin_1 = find_tone_bin(CW_TONE_1_MHZ, N_CAP, ADC_SR)
    if bin_0 == bin_1:
        raise ValueError("both tones land on FFT bin %d - they must differ by at least "
                         "one bin of the %g Hz grid" % (bin_0, ADC_SR / N_CAP))
    print("side 0: %.6f MHz on bin %d | side 1: %.6f MHz on bin %d | %d bins apart"
          % (CW_TONE_0_MHZ, bin_0, CW_TONE_1_MHZ, bin_1, abs(bin_1 - bin_0)))

def validate_bsic_settings():
    """The stage 2 knob and the hardware limits are sane, and say what they cost."""
    if REGULARIZATION <= 0:
        raise ValueError("REGULARIZATION must be positive, got %g" % REGULARIZATION)
    if not 0 < BSIC_MAX_GAIN <= 2.0:
        raise ValueError("BSIC_MAX_GAIN must be between 0 and 2.0, got %g"
                         % BSIC_MAX_GAIN)
    if not 90.0 <= MAX_PHASE_DEG < 180.0:
        raise ValueError("MAX_PHASE_DEG must be just under 180, got %g" % MAX_PHASE_DEG)
    probes = len(DACS_0) + len(DACS_1)
    print("bsic: two stage, regularization %g - "
          "%d probe captures, 2 solo captures, 2 test captures"
          % (REGULARIZATION, probes))

validate_systems()
validate_rf_settings()
CW_TONE_0_MHZ = snap_tone_to_fft_bin(CW_TONE_0_MHZ, ADC_SR, N_CAP)
CW_TONE_1_MHZ = snap_tone_to_fft_bin(CW_TONE_1_MHZ, ADC_SR, N_CAP)
validate_tone_separation()
validate_bsic_settings()

print("parameters ok - side 0: DAC %s ADC %s | side 1: DAC %s ADC %s"
      % (DACS_0, ADCS_0, DACS_1, ADCS_1))

## 3. Set up the board

Each side's tone goes into **its own tile's player memory** once and is never
rewritten. All four DACs of a tile read that player, so the gain and phase table
alone decides which elements are on and how they combine. Every capture in this
notebook - probe, solo or test - transmits the same two waveforms with the same
phase, which is what lets a column measured in one capture sit next to a column
measured in the next.

Muting a DAC is a gain of zero in its table, not a rewritten player.

`capture_aligned()` fires the one trigger edge that starts every transmitter and
every receiver together, so the phase of what comes back is a property of the
path and not of when the trigger fired.

In [ ]:
from lib.mts import doaMtsOverlay

SIDES = {0: {"dacs": DACS_0, "adcs": ADCS_0, "far": 1,
             "own_tone": CW_TONE_0_MHZ, "far_tone": CW_TONE_1_MHZ},
         1: {"dacs": DACS_1, "adcs": ADCS_1, "far": 0,
             "own_tone": CW_TONE_1_MHZ, "far_tone": CW_TONE_0_MHZ}}

def setup_board():
    """Load the overlay, tune both converter sets, and give each side its own tone."""
    overlay = doaMtsOverlay("mts.bit")
    tune_dacs(overlay, DAC_NCO, DAC_ZONE, N_CH)
    tune_adcs(overlay, ADC_NCO, ADC_ZONE, N_CH, N_CAP)
    n_samples = overlay.dac0_player.shape[0] // 2
    for side in SIDES:
        tile = SIDES[side]["dacs"][0] // PATH_PER_TILE
        tone_mhz = SIDES[side]["own_tone"]
        tone = create_tone_samples(n_samples, DAC_SR, tone_mhz * 1e6, CW_AMP)
        write_tone_to_tile_player(overlay, tone, tile)
        print("side %d: DAC tile %d loaded with the %.6f MHz tone" % (side, tile, tone_mhz))
    return overlay

def capture_block(overlay, gains, phases):
    """Drive the DACs with one gain and phase table and take one block, TX and RX together."""
    overlay.d_gain = gains
    overlay.d_phases = phases
    overlay.configure_dacs()
    overlay.da = 2
    raw = capture_aligned(overlay, TRIG_HOLD_S)
    iq = convert_raw_to_iq(raw, N_CH)
    if iq.shape[1] < N_CAP:
        raise ValueError("capture is %d samples long, N_CAP asks for %d"
                         % (iq.shape[1], N_CAP))
    return iq[:, :N_CAP]

def create_tx_reference(tone_mhz):
    """One side's transmitted tone at the receive rate, unit power, phase zero at the trigger."""
    time_axis = np.arange(N_CAP) / ADC_SR
    return np.exp(2j * np.pi * tone_mhz * 1e6 * time_axis)

TX_REFERENCE = {0: create_tx_reference(CW_TONE_0_MHZ),
                1: create_tx_reference(CW_TONE_1_MHZ)}

OVERLAY = setup_board()
print("board ready: %d ADC channels open" % N_CH)

## 4. Weights as the hardware takes them

Three kinds of gain and phase table are used below:

| table | who is on | used by |
|---|---|---|
| probe | one DAC of one side, at the ceiling | first training |
| solo | one side with its transmit weight, the other muted | second training |
| both | both sides with their transmit weights | the test |

The transmit weights are scaled to `BSIC_MAX_GAIN` on each side's **own** peak
element, because a beam lives in the ratios between elements and not in their
absolute level. The receive weights never touch the hardware - they are a complex
sum in numpy after the capture.

In [ ]:
def create_hardware_weights(weights):
    """Both transmit vectors as the gain and phase tables the DACs take."""
    gains = [0.0] * N_CH
    phases = [0.0] * N_CH
    for side in SIDES:
        vector = weights["tx_%d" % side]
        peak = np.max(np.abs(vector))
        for element, dac in enumerate(SIDES[side]["dacs"]):
            weight = vector[element] * BSIC_MAX_GAIN / peak
            degrees = (np.rad2deg(np.angle(weight)) + 180.0) % 360.0 - 180.0
            gains[dac] = abs(weight)
            phases[dac] = max(-MAX_PHASE_DEG, min(MAX_PHASE_DEG, degrees))
    return gains, phases

def create_silent_gains(gains, dacs):
    """The same gain table with one side muted, for a capture it must not be in."""
    silent = list(gains)
    for dac in dacs:
        silent[dac] = 0.0
    return silent

def create_probe_tables(side, element):
    """One DAC of one side at the ceiling and everything else muted - a stage 1 probe."""
    gains = [0.0] * N_CH
    phases = [0.0] * N_CH
    gains[SIDES[side]["dacs"][element]] = BSIC_MAX_GAIN
    return gains, phases

def create_uniform_weights():
    """Every element driven and heard equally - the reference every number is read against."""
    weights = {}
    for side in SIDES:
        elements = len(SIDES[side]["dacs"])
        receivers = len(SIDES[side]["adcs"])
        weights["tx_%d" % side] = np.ones(elements, dtype=complex) / np.sqrt(elements)
        weights["rx_%d" % side] = np.ones(receivers, dtype=complex) / np.sqrt(receivers)
    return weights

print("weight tables ready")

## 5. Stage 1 - first training, the transmit weights

Each side probes its own self interference channel **one DAC at a time**, the
other side muted. Each probe drives one column of `H_si`; correlating every ADC's
block against the tone that was sent reads that column back. The columns side by
side are the channel, and its gram matrix `H^H H` says how much of every transmit
direction reaches the side's own receivers.

```
tx weight = eigenvector of H^H H with the smallest eigenvalue
```

`eigh` sorts upwards, so that is column 0. No conjugate on the way out: `w^H H^H H w`
already is the power `||H w||^2` the weight sends.

The probes are one DAC at a time and not one uniform capture because a single
weight only shows one direction - the quietest one could hide in what it misses.

The eigenvalues printed here are the ceiling on what this stage can do: the ratio
of the largest to the smallest is how much quieter the quietest direction is than
the loudest. All of them within a few dB means nowhere quiet to stand.

In [ ]:
def run_first_training(overlay):
    """Each side's self interference gram matrix, probed one DAC at a time."""
    correlations = {}
    for side in SIDES:
        correlations[side] = measure_tx_correlation(overlay, side)
    return correlations

def measure_tx_correlation(overlay, side):
    """H^H H of one side's self interference channel, read off its probes.

    The reference is unit power, so a column carries the DAC amplitude inside it.
    That is a common scale on every column and moves no eigenvector.
    """
    adcs = SIDES[side]["adcs"]
    reference = TX_REFERENCE[side]
    columns = []
    for element in range(len(SIDES[side]["dacs"])):
        gains, phases = create_probe_tables(side, element)
        iq = capture_block(overlay, gains, phases)
        column = iq[adcs] @ reference.conj() / N_CAP
        columns.append(column)
        print("  side %d probe DAC %2d: |h| %s"
              % (side, SIDES[side]["dacs"][element], np.round(np.abs(column), 1)))
    measured_channel = np.column_stack(columns)
    return measured_channel.conj().T @ measured_channel

def calculate_tx_weights(correlations):
    """Stage 1, each side transmits down the direction its own coupling hears least."""
    weights = {}
    for side in SIDES:
        weights["tx_%d" % side] = solve_quietest_tx_direction(correlations[side])
    return weights

def solve_quietest_tx_direction(tx_correlation):
    """The transmit weight that puts the least power into the channel."""
    _, eigenvectors = np.linalg.eigh(tx_correlation)
    return eigenvectors[:, 0]

def describe_tx_correlation(side, tx_correlation):
    """The eigenvalues of one side's H^H H, and the headroom between the ends."""
    eigenvalues = np.maximum(np.linalg.eigvalsh(tx_correlation), 0.0)
    quietest = max(eigenvalues[0], eigenvalues[-1] * 1e-12)
    print("side %d H^H H eigenvalues: %s"
          % (side, np.array2string(eigenvalues, precision=3)))
    print("side %d stage 1 headroom: about %.1f dB between its quietest direction "
          "and its loudest" % (side, 10 * np.log10(eigenvalues[-1] / quietest)))

TX_CORRELATIONS = run_first_training(OVERLAY)
TX_WEIGHTS = calculate_tx_weights(TX_CORRELATIONS)
for side in SIDES:
    describe_tx_correlation(side, TX_CORRELATIONS[side])
    print("side %d tx weight: %s" % (side, np.round(TX_WEIGHTS["tx_%d" % side], 4)))

## 6. Stage 2 - second training, the receive weights

Each side transmits **alone**, with its stage 1 weight on air. One solo capture
serves two matrices at once - the transmitting side's own ADCs hold what still
leaks, the other side's ADCs hold the link:

| solo capture | its own ADCs give | the other side's ADCs give |
|---|---|---|
| side 0 alone | `si_0` - what side 0 still leaks on itself | `link_1` - what side 1 hears from side 0 |
| side 1 alone | `si_1` | `link_0` |

Every matrix is the spatial covariance `R = mean(y y^H)` on the raw antennas.
Measured **after** stage 1 and not before, so `si` holds only the one direction
the transmit weight could not avoid, and `link` only the one direction the far
side now points at.

```
rx weight = conj( strongest generalized eigenvector of (R_si + lambda I)^-1 R_link )
```

Conjugated because the eigenvector maximizes `v^H R v` while the weight meets the
antennas as `w^T y`; `w = v*` is what gives the same ratio.

`lambda` is `REGULARIZATION`, added as is to the self interference covariance.
That covariance is close to rank one, so without it the inverse would blow up on
the noise; with too much of it the weight stops caring about the null and just
chases the link. It is an absolute number in ADC counts squared, on the same
scale as the covariance traces printed by the solo captures - read those before
picking it.

In [ ]:
def run_second_training(overlay, tx_weights):
    """The four receive covariances, out of one solo capture per side."""
    gains, phases = create_hardware_weights(tx_weights)
    correlations = {}
    for side in SIDES:
        far = SIDES[side]["far"]
        solo_gains = create_silent_gains(gains, SIDES[far]["dacs"])
        iq = capture_block(overlay, solo_gains, phases)
        correlations["si_%d" % side] = measure_rx_correlation(iq[SIDES[side]["adcs"]])
        correlations["link_%d" % far] = measure_rx_correlation(iq[SIDES[far]["adcs"]])
        print("  side %d alone: si_%d trace %.4e   link_%d trace %.4e"
              % (side, side, np.trace(correlations["si_%d" % side]).real,
                 far, np.trace(correlations["link_%d" % far]).real))
    return correlations

def measure_rx_correlation(antennas):
    """The spatial covariance on a set of receive antennas, the time average of y y^H."""
    return antennas @ antennas.conj().T / antennas.shape[1]

def calculate_rx_weights(correlations):
    """Stage 2, each side receives for the best sinr against what is actually on air."""
    weights = {}
    for side in SIDES:
        weights["rx_%d" % side] = solve_max_sinr_rx_direction(
            correlations["link_%d" % side], correlations["si_%d" % side])
    return weights

def solve_max_sinr_rx_direction(link_correlation, si_correlation):
    """The receive weight with the most link power per unit of leakage plus the floor."""
    regularized = si_correlation + REGULARIZATION * np.eye(len(si_correlation))
    return np.conj(find_max_generalized_eigenvector(link_correlation, regularized))

RX_CORRELATIONS = run_second_training(OVERLAY, TX_WEIGHTS)
RX_WEIGHTS = calculate_rx_weights(RX_CORRELATIONS)
BSIC_WEIGHTS = dict(TX_WEIGHTS)
BSIC_WEIGHTS.update(RX_WEIGHTS)
for side in SIDES:
    print("side %d rx weight: %s" % (side, np.round(RX_WEIGHTS["rx_%d" % side], 4)))

## 7. Test - both sides on air at once

Two captures with everything transmitting: one on uniform weights, the reference,
and one on the weights the two stages produced.

```
side 0 tone --+--> H00 --> side 0 ADCs   its own leakage
              +--> H10 --> side 1 ADCs   the link, wanted at side 1
side 1 tone --+--> H11 --> side 1 ADCs   its own leakage
              +--> H01 --> side 0 ADCs   the link, wanted at side 0
```

Each side combines its ADCs with its receive weight, takes one FFT, and reads two
bins: its own tone is the leakage, the other side's tone is the link. The power on
its raw antennas before combining is what its amplifier had to survive.

In [ ]:
def receive_both_sides(overlay, weights, label):
    """One capture with both sides on air, read per side."""
    gains, phases = create_hardware_weights(weights)
    iq = capture_block(overlay, gains, phases)
    result = {}
    for side in SIDES:
        result[side] = receive_one_side(iq, weights, side)
    print("  %-28s side 0 si %.4e link %.4e | side 1 si %.4e link %.4e"
          % (label, result[0]["si_amplitude"], result[0]["link_amplitude"],
             result[1]["si_amplitude"], result[1]["link_amplitude"]))
    return result

def receive_one_side(iq, weights, side):
    """The two tone amplitudes one receiver reads after its weight, and its antenna power before it."""
    antennas = iq[SIDES[side]["adcs"]]
    combined = weights["rx_%d" % side] @ antennas
    spectrum = np.fft.fft(combined) / N_CAP
    return {"si_amplitude": read_tone_amplitude(spectrum, SIDES[side]["own_tone"]),
            "link_amplitude": read_tone_amplitude(spectrum, SIDES[side]["far_tone"]),
            "antenna_power": np.sum(np.abs(antennas) ** 2) / N_CAP,
            "antennas": antennas,
            "combined": combined}

def read_tone_amplitude(spectrum, tone_mhz):
    """The amplitude of one tone, read off its own FFT bin, floored so a null stays finite."""
    tone_bin = find_tone_bin(tone_mhz, len(spectrum), ADC_SR)
    return max(np.abs(spectrum[tone_bin]), 1e-15)

BEFORE = receive_both_sides(OVERLAY, create_uniform_weights(), "uniform, the reference")
AFTER = receive_both_sides(OVERLAY, BSIC_WEIGHTS, "two stage bsic")

## 8. Analyze the results

Per side, what the weights bought:

- **Cancellation depth** - own tone before over after, squared. Positive is
  cancellation.
- **Link preservation** - the other side's tone after over before. Negative means
  the weights damaged the wanted signal.
- **SINR before / after** - link over leakage inside the same capture. The number a
  real receiver cares about, and the one depth and preservation are the two
  halves of.
- **Antenna power before / after** - summed over the side's raw ADCs, in dB against
  one ADC count. The change is stage 1's payoff: how much less its amplifier has
  to survive. Stage 2 cannot move this, it runs after the converter.

Nothing is averaged - every number is one block.

In [ ]:
RESULT_HEADERS = ["depth_db", "preservation_db", "sinr_before_db", "sinr_after_db",
                  "antenna_before_db", "antenna_after_db", "antenna_change_db"]

def calculate_measurements(before, after):
    """One row per side, all in dB."""
    rows = []
    for side in SIDES:
        rows.append(calculate_side_measurements(side, before[side], after[side]))
    return rows

def calculate_side_measurements(side, before, after):
    """The numbers of one side, every ratio quoted against the uniform capture."""
    antenna_before_db = 10 * np.log10(before["antenna_power"])
    antenna_after_db = 10 * np.log10(after["antenna_power"])
    return {"side": side,
            "depth_db": 20 * np.log10(before["si_amplitude"] / after["si_amplitude"]),
            "preservation_db": 20 * np.log10(after["link_amplitude"] / before["link_amplitude"]),
            "sinr_before_db": 20 * np.log10(before["link_amplitude"] / before["si_amplitude"]),
            "sinr_after_db": 20 * np.log10(after["link_amplitude"] / after["si_amplitude"]),
            "antenna_before_db": antenna_before_db,
            "antenna_after_db": antenna_after_db,
            "antenna_change_db": antenna_after_db - antenna_before_db}

def print_result_table(rows):
    """One row per side, the metrics as columns."""
    print("two stage bsic, regularization %g" % REGULARIZATION)
    print("%-6s" % "side" + "".join("%19s" % header for header in RESULT_HEADERS))
    for row in rows:
        print("%-6d" % row["side"] + "".join("%19.2f" % row[header]
                                             for header in RESULT_HEADERS))

RESULTS = calculate_measurements(BEFORE, AFTER)
print_result_table(RESULTS)

## 9. Plot

Each side's combined spectrum before and after, with both tone bins marked: red
is the side's own tone, the one that should sink, green is the link, the one that
should not move. Then the table as bars.

In [ ]:
import matplotlib.pyplot as plt

def plot_spectra(before, after):
    """Both sides' combined streams, uniform against bsic, both tone bins marked."""
    frequency_mhz = np.fft.fftshift(np.fft.fftfreq(N_CAP, 1 / ADC_SR)) / 1e6
    figure, axes = plt.subplots(1, 2, figsize=(13, 4))
    for side in SIDES:
        axis = axes[side]
        for label, result in (("uniform", before[side]), ("two stage bsic", after[side])):
            spectrum = np.fft.fftshift(np.fft.fft(result["combined"])) / N_CAP
            power_db = 10 * np.log10(np.abs(spectrum) ** 2 + 1e-30)
            axis.plot(frequency_mhz, power_db, label=label, linewidth=1)
        own_tone = SIDES[side]["own_tone"]
        far_tone = SIDES[side]["far_tone"]
        axis.axvline(own_tone, color="tab:red", linestyle="--", linewidth=1)
        axis.axvline(far_tone, color="tab:green", linestyle="--", linewidth=1)
        axis.set_xlim(min(own_tone, far_tone) - 1, max(own_tone, far_tone) + 1)
        axis.set_xlabel("baseband frequency [MHz]")
        axis.set_ylabel("power [dB]")
        axis.set_title("side %d - red: its own tone, green: the link" % side)
        axis.grid(alpha=0.3)
        axis.legend()
    figure.tight_layout()

def plot_summary(rows):
    """Depth, preservation, sinr after and antenna change, the two sides side by side."""
    panels = [("cancellation depth [dB]", "depth_db"),
              ("link preservation [dB]", "preservation_db"),
              ("sinr after [dB]", "sinr_after_db"),
              ("antenna power change [dB]", "antenna_change_db")]
    figure, axes = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 4))
    for axis, (title, key) in zip(axes, panels):
        labels = ["side %d" % row["side"] for row in rows]
        values = [row[key] for row in rows]
        axis.bar(labels, values, color=["tab:blue", "tab:orange"])
        axis.set_title(title)
        axis.grid(axis="y", alpha=0.3)
        for position, value in enumerate(values):
            axis.text(position, value, "%.1f" % value, ha="center", fontsize=8,
                      va="bottom" if value >= 0 else "top")
    figure.tight_layout()

plot_spectra(BEFORE, AFTER)
plot_summary(RESULTS)
plt.show()

## 10. Save

| file | what it holds |
|---|---|
| `params.json` | every setting the run used |
| `results.md` | the table, the transmit tables per DAC and the receive weights per ADC |
| `training.npz` | the two `H^H H` matrices and the four covariances the weights came from |
| `signals.npz` | each side's raw antennas and combined stream, before and after |

The folder is emptied first, so nothing stale reads as a result of this run. Only
the files sitting directly in it.

In [ ]:
def save_run(rows, weights, before, after):
    """Replace the folder with the settings, the report, the training and the signals."""
    clear_dir(SIC_DIR)
    save_params(SIC_DIR)
    save_results_report(SIC_DIR, rows, weights)
    save_training(SIC_DIR)
    save_signals(SIC_DIR, before, after)
    print("saved %s: %s" % (SIC_DIR, sorted(os.listdir(SIC_DIR))))

def save_params(folder):
    """Record the settings the numbers came from, so a result can be traced back."""
    params = {"dacs_0": DACS_0, "adcs_0": ADCS_0,
              "dacs_1": DACS_1, "adcs_1": ADCS_1,
              "dac_nco": DAC_NCO, "dac_zone": DAC_ZONE,
              "adc_nco": ADC_NCO, "adc_zone": ADC_ZONE,
              "cw_tone_0_mhz": CW_TONE_0_MHZ, "cw_tone_1_mhz": CW_TONE_1_MHZ,
              "cw_amp": CW_AMP,
              "regularization": REGULARIZATION,
              "bsic_max_gain": BSIC_MAX_GAIN,
              "max_phase_deg": MAX_PHASE_DEG,
              "n_cap": N_CAP, "dac_sr": DAC_SR, "adc_sr": ADC_SR}
    save_json_params(folder, params)

def save_results_report(folder, rows, weights):
    """The table and every weight behind it, as the markdown that goes in the writeup."""
    gains, phases = create_hardware_weights(weights)
    lines = ["# Two stage BSIC run, no channel estimate", "",
             "Regularization %g, in ADC counts squared." % REGULARIZATION, "",
             "Side 0 transmits %.6f MHz, side 1 transmits %.6f MHz."
             % (CW_TONE_0_MHZ, CW_TONE_1_MHZ), "",
             "## Results", "",
             "| side | " + " | ".join(RESULT_HEADERS) + " |",
             "|---" * (len(RESULT_HEADERS) + 1) + "|"]
    for row in rows:
        values = " | ".join("%.2f" % row[header] for header in RESULT_HEADERS)
        lines.append("| %d | %s |" % (row["side"], values))
    for side in SIDES:
        lines.append("")
        lines.append("## Side %d" % side)
        lines.append("")
        lines.append("Transmit, per DAC:")
        lines.append("")
        lines.append("| DAC | gain | phase [deg] |")
        lines.append("|---|---|---|")
        for dac in SIDES[side]["dacs"]:
            lines.append("| %d | %.4f | %+.3f |" % (dac, gains[dac], phases[dac]))
        lines.append("")
        lines.append("Receive combine, per ADC:")
        lines.append("")
        lines.append("| ADC | magnitude | phase [deg] |")
        lines.append("|---|---|---|")
        for adc, value in zip(SIDES[side]["adcs"], weights["rx_%d" % side]):
            lines.append("| %d | %.4f | %+.3f |"
                         % (adc, abs(value), np.degrees(np.angle(value))))
    with open(os.path.join(folder, "results.md"), "w") as handle:
        handle.write("\n".join(lines) + "\n")

def save_training(folder):
    """Every matrix the weights were solved on."""
    np.savez(os.path.join(folder, "training.npz"),
             tx_correlation_0=TX_CORRELATIONS[0],
             tx_correlation_1=TX_CORRELATIONS[1],
             **RX_CORRELATIONS)

def save_signals(folder, before, after):
    """Each side's raw antennas and combined stream, on uniform weights and on the bsic ones."""
    arrays = {}
    for label, result in (("before", before), ("after", after)):
        for side in SIDES:
            arrays["antennas_%s_%d" % (label, side)] = result[side]["antennas"]
            arrays["combined_%s_%d" % (label, side)] = result[side]["combined"]
    np.savez(os.path.join(folder, "signals.npz"), **arrays)

save_run(RESULTS, BSIC_WEIGHTS, BEFORE, AFTER)

## 11. Stop

Switch both transmitters off when you are done.

In [ ]:
OVERLAY.dacs_off()
print("dacs off")